In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import chess
import ipywidgets as widgets

from beschess.data.embedding import BalancedBatchSampler, PuzzleDataset, generate_split_indices
from beschess.components.net.vit import get_interpretable_vit, extract_attention_weights
from beschess.viz import plot_chessboard_attention_overlay
from beschess.utils import tensor_to_board

%load_ext autoreload
%autoreload 2

In [2]:
data_dir = '../data/processed/'
quiet_boards_file = data_dir + 'quiet_boards_preeval.npy'
puzzle_boards_file = data_dir + 'boards_packed.npy'
puzzle_labels_file = data_dir + 'tags_packed.npy'

quiet_boards = np.load(quiet_boards_file, mmap_mode='r')
puzzle_boards = np.load(puzzle_boards_file, mmap_mode='r')
puzzle_labels = np.load(puzzle_labels_file, mmap_mode='r')

dataset = PuzzleDataset(
    quiet_boards=quiet_boards,
    puzzle_boards=puzzle_boards,
    puzzle_labels=puzzle_labels,
)

splits = generate_split_indices(dataset)
q_test, p_test = splits['test']

dataloader = DataLoader(
    dataset,
    batch_sampler=BalancedBatchSampler(
        dataset, 
        q_test, 
        p_test,
        batch_size=4,
    ),
)

Building label map: 100%|██████████| 208521/208521 [00:01<00:00, 199191.81it/s]


In [3]:
checkpoint_path = "../checkpoints/OptimizedModule_20251203_151336/best_checkpoint.pth"
model = get_interpretable_vit(
    in_channels=17,
    embed_dim=256,
    num_heads=8,
    depth=6,
    out_dim=128,
    path_to_weights=checkpoint_path,
)

In [4]:
for batch in dataloader:
    inputs, labels = batch
    embeddings, puzzle_probs, attn_weights = extract_attention_weights(model, inputs)
    break

In [5]:
def compute_attention_rollout(all_layer_weights):
    # all_layer_weights shape: (Layers, Batch, Heads, Seq, Seq)
    
    # 1. Average over heads (simplify to single matrix per layer)
    # Shape: (Layers, Seq, Seq)
    avg_weights = all_layer_weights[0].mean(dim=1) 

    # 2. Initialize "Rollout" with Identity Matrix
    rollout = torch.eye(avg_weights.shape[1])
    
    # 3. Multiply matrices recursively
    # This propagates the attention flow from Layer 0 up to N
    for i, layer_attn in enumerate(all_layer_weights):
        # We add Identity to account for "Residual Connections" 
        # (The token remembering itself)
        layer_attn_avg = layer_attn.mean(dim=1) # Average heads
        identity = torch.eye(layer_attn_avg.shape[1])
        
        # A_hat = 0.5 * A + 0.5 * I (Heuristic for residual weight)
        fused = 0.5 * layer_attn_avg + 0.5 * identity
        
        # Multiply: New_Rollout = Old_Rollout @ New_Layer
        rollout = torch.matmul(rollout, fused)
        
    return rollout

In [6]:
rollout = compute_attention_rollout(attn_weights).unsqueeze(1)
rollout.shape

torch.Size([6, 1, 65, 65])

In [10]:
batch_idx = 0
board = tensor_to_board(inputs[batch_idx].numpy())
print(board.fen())
w = widgets.interact(
    plot_chessboard_attention_overlay,
    board=widgets.fixed(board),
    attention_map=widgets.fixed(attn_weights[batch_idx].numpy()),
    depth=widgets.IntSlider(min=0, max=6-1, step=1, value=0),
    query_idx=widgets.IntSlider(min=0, max=65-1, step=1, value=0),
    head_idx=widgets.IntSlider(min=0, max=8-1, step=1, value=0),


(17, 8, 8)
rnb2rk1/pp3ppp/2p1pb2/7Q/4N3/P2B4/1qP2PPP/R3K1NR w KQ - 0 1


interactive(children=(IntSlider(value=0, description='query_idx', max=64), IntSlider(value=0, description='hea…

In [ ]:
tensor = inputs[batch_idx].numpy()
piece_only = tensor[:12, :, :]
piece_only.sum(axis=0)

(17, 8, 8)


array([[1., 0., 0., 0., 1., 0., 1., 1.],
       [0., 1., 1., 0., 0., 1., 1., 1.],
       [1., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 1., 1., 0., 0.],
       [1., 1., 0., 0., 0., 1., 1., 1.],
       [1., 1., 1., 0., 0., 1., 1., 0.]], dtype=float32)